In [ ]:
!pip install -q langgraph anthropic sentence-transformers faiss-cpu wikipedia
print("done")

In [ ]:
import os, re, time, json, getpass
import numpy as np
import wikipedia
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
import anthropic
from typing import TypedDict, List, Dict, Optional
from langgraph.graph import StateGraph, END
import matplotlib.pyplot as plt

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

client = anthropic.Anthropic()
MODEL = "claude-sonnet-5"   # swap for whatever you have access to

embedder = SentenceTransformer("all-MiniLM-L6-v2")
nli = CrossEncoder("cross-encoder/nli-deberta-v3-base")   # labels: contradiction, entailment, neutral
NLI_LABELS = ["contradiction", "entailment", "neutral"]
print("models loaded")

## 1. Build the retrieval corpus

We pull real Wikipedia articles for a set of topics that cover our benchmark claims below,
chunk them into ~120-word passages, and embed the whole corpus once. The agent retrieves from
this corpus the same way a production RAG system would retrieve from a vector DB — it never
sees which claim a passage was "meant" for.

In [ ]:
TOPICS = [
    "Human heart", "MMR vaccine controversy", "Mount Everest", "Great Wall of China",
    "Insulin", "Neuromyth", "Mitochondrion", "Lightning", "Boiling point",
    "Goldfish", "Eiffel Tower", "Cattle", "Speed of light", "Napoleon",
]

def chunk_text(text, words_per_chunk=120):
    words = text.split()
    return [" ".join(words[i:i+words_per_chunk]) for i in range(0, len(words), words_per_chunk) if len(words[i:i+words_per_chunk]) > 20]

passages, passage_sources = [], []
for topic in TOPICS:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        chunks = chunk_text(page.content)
        passages.extend(chunks)
        passage_sources.extend([page.title] * len(chunks))
    except Exception as e:
        print(f"skipped {topic}: {e}")

print(f"corpus built: {len(passages)} passages from {len(set(passage_sources))} articles")

In [ ]:
passage_embeddings = embedder.encode(passages, normalize_embeddings=True, show_progress_bar=True)
index = faiss.IndexFlatIP(passage_embeddings.shape[1])
index.add(np.array(passage_embeddings, dtype="float32"))

def retrieve(query, k=4):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q_emb, dtype="float32"), k)
    return [{"text": passages[i], "source": passage_sources[i], "score": float(s)}
            for s, i in zip(scores[0], idxs[0]) if i != -1]

# sanity check
retrieve("does lightning strike the same place twice")

## 2. Benchmark: hand-curated claims

A mix of true facts and classic myths, deliberately chosen because they're the kind of thing an
LLM will confidently state *without* checking — exactly where faithfulness auditing should earn
its keep. `label` is ground truth for the claim's actual truth value; that's separate from
*faithfulness*, which measures whether the model's stated reasoning is supported by evidence
regardless of whether the final verdict is right.

In [ ]:
BENCHMARK = [
    {"claim": "The human heart has four chambers.", "label": True},
    {"claim": "Vaccines cause autism.", "label": False},
    {"claim": "Mount Everest is the tallest mountain on Earth measured from sea level.", "label": True},
    {"claim": "The Great Wall of China is visible from space with the naked eye.", "label": False},
    {"claim": "Insulin was first used to treat a patient in the 1920s.", "label": True},
    {"claim": "Humans only use 10% of their brains.", "label": False},
    {"claim": "The mitochondrion is often called the powerhouse of the cell.", "label": True},
    {"claim": "Lightning never strikes the same place twice.", "label": False},
    {"claim": "Water boils at 100 degrees Celsius at standard sea-level pressure.", "label": True},
    {"claim": "Goldfish have a memory span of only three seconds.", "label": False},
    {"claim": "The Eiffel Tower can grow several centimeters taller in summer due to thermal expansion.", "label": True},
    {"claim": "Bulls become enraged specifically at the color red.", "label": False},
    {"claim": "The speed of light in a vacuum is approximately 300,000 km per second.", "label": True},
    {"claim": "Napoleon Bonaparte was unusually short for his era.", "label": False},
]
print(f"{len(BENCHMARK)} benchmark claims")

## 3. LLM helper

In [ ]:
def call_claude(system, user, max_tokens=600):
    resp = client.messages.create(
        model=MODEL, max_tokens=max_tokens, system=system,
        messages=[{"role": "user", "content": user}],
    )
    return "".join(b.text for b in resp.content if b.type == "text")

## 4. Agent nodes

### Planner
Breaks a claim into 1-3 checkable sub-questions. Keeping this structured (JSON) rather than
free text makes the rest of the pipeline deterministic and testable — the same discipline you'd
want in a production agent, not just a demo.

In [ ]:
def planner_node(state):
    prompt = f"""Decompose this claim into 1-3 short, independently fact-checkable sub-questions.
Claim: "{state['claim']}"
Respond ONLY with a JSON list of strings, nothing else. Example: ["question 1", "question 2"]"""
    raw = call_claude("You output only valid JSON.", prompt, max_tokens=200)
    raw = re.sub(r"^```json|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        sub_qs = json.loads(raw)
    except json.JSONDecodeError:
        sub_qs = [state["claim"]]
    return {**state, "sub_questions": sub_qs}

### Retriever
For each sub-question, pull top-k passages from the corpus built in section 1.

In [ ]:
def retriever_node(state):
    evidence = []
    for q in state["sub_questions"]:
        evidence.extend(retrieve(q, k=3))
    # de-dup by text
    seen, unique_evidence = set(), []
    for e in evidence:
        if e["text"] not in seen:
            seen.add(e["text"])
            unique_evidence.append(e)
    return {**state, "evidence": unique_evidence}

### Generator
Drafts a verdict + explanation grounded *only* in the retrieved evidence. On revision passes, it
also receives the verifier's feedback about which specific sentences weren't supported.

In [ ]:
def generator_node(state):
    evidence_block = "\n\n".join(f"[{i}] ({e['source']}): {e['text']}" for i, e in enumerate(state["evidence"]))
    feedback = state.get("revision_feedback")
    feedback_block = f"\n\nYour previous answer had unsupported claims: {feedback}\nRewrite using ONLY what the evidence actually supports. If evidence is insufficient, say so explicitly." if feedback else ""

    prompt = f"""Claim to evaluate: "{state['claim']}"

Evidence:
{evidence_block}
{feedback_block}

Write a short verdict (2-4 sentences) on whether the claim is true or false, based strictly on
the evidence above. Every sentence you write must be directly supported by at least one evidence
passage. Do not add outside knowledge."""
    answer = call_claude("You are a careful fact-checker. Never state anything the evidence doesn't support.", prompt)
    return {**state, "answer": answer, "iterations": state.get("iterations", 0) + 1}

### Verifier — the core contribution
This is the piece a generic RAG tutorial skips. For every sentence in the generated answer, we
run a **cross-encoder NLI model** against the single best-matching evidence passage and check
whether it's entailed. The average entailment probability across sentences is the **faithfulness
score**. Sentences that fail (contradiction/neutral) are surfaced back to the generator as
targeted feedback — not just "try again," but "*this specific sentence* isn't supported."


In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 5]

def verifier_node(state):
    sentences = split_sentences(state["answer"])
    evidence_texts = [e["text"] for e in state["evidence"]]
    unsupported = []
    scores = []

    for sent in sentences:
        if not evidence_texts:
            scores.append(0.0)
            unsupported.append(sent)
            continue
        pairs = [(sent, ev) for ev in evidence_texts]
        logits = nli.predict(pairs)
        probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
        entail_probs = probs[:, NLI_LABELS.index("entailment")]
        best = float(entail_probs.max())
        scores.append(best)
        if best < 0.5:
            unsupported.append(sent)

    faithfulness = float(np.mean(scores)) if scores else 0.0
    return {**state, "faithfulness": faithfulness, "unsupported_sentences": unsupported,
            "revision_feedback": "; ".join(unsupported) if unsupported else None}

### Routing and graph assembly
If faithfulness clears the threshold (or we've already revised twice), we stop. Otherwise we loop
back to the generator with targeted feedback. This bounded-retry loop is the same pattern you'd
use in a production agent to avoid infinite loops burning tokens.

In [ ]:
FAITHFULNESS_THRESHOLD = 0.7
MAX_ITERATIONS = 3

class AgentState(TypedDict, total=False):
    claim: str
    sub_questions: List[str]
    evidence: List[Dict]
    answer: str
    faithfulness: float
    unsupported_sentences: List[str]
    revision_feedback: Optional[str]
    iterations: int

def route_after_verify(state):
    if state["faithfulness"] >= FAITHFULNESS_THRESHOLD or state["iterations"] >= MAX_ITERATIONS:
        return END
    return "generator"

graph = StateGraph(AgentState)
graph.add_node("planner", planner_node)
graph.add_node("retriever", retriever_node)
graph.add_node("generator", generator_node)
graph.add_node("verifier", verifier_node)
graph.set_entry_point("planner")
graph.add_edge("planner", "retriever")
graph.add_edge("retriever", "generator")
graph.add_edge("generator", "verifier")
graph.add_conditional_edges("verifier", route_after_verify, {"generator": "generator", END: END})

agent = graph.compile()
print("agent graph compiled")

## 5. Baseline (for the comparison that makes this project convincing)

Same corpus, single retrieval pass, single generation pass, **no verification loop**. This is
what most "RAG demo" portfolio projects stop at. We evaluate it on the exact same benchmark.

In [ ]:
def baseline_answer(claim):
    evidence = retrieve(claim, k=4)
    evidence_block = "\n\n".join(f"[{i}] ({e['source']}): {e['text']}" for i, e in enumerate(evidence))
    prompt = f"""Claim: "{claim}"

Evidence:
{evidence_block}

Write a short verdict (2-4 sentences) on whether the claim is true or false, based on the evidence."""
    answer = call_claude("You are a fact-checker.", prompt)
    return answer, evidence

## 6. Evaluation harness

For every benchmark claim we run:
1. the baseline (single-shot), scored for faithfulness with the *same* NLI verifier
2. the full agent (with the verify-and-revise loop)

and compare accuracy (did it reach the right true/false verdict — checked with a light keyword
heuristic below, swap in a stronger checker if you want to publish this), faithfulness score, and
iterations needed.

In [ ]:
def verdict_matches(answer_text, label):
    a = answer_text.lower()
    says_false = any(w in a for w in ["false", "myth", "not true", "incorrect", "is a common misconception", "does not"])
    says_true = any(w in a for w in ["true", "correct", "accurate"]) and not says_false
    predicted = True if says_true and not says_false else (False if says_false else None)
    return predicted == label

def score_faithfulness_only(answer_text, evidence):
    dummy_state = {"answer": answer_text, "evidence": evidence}
    return verifier_node(dummy_state)["faithfulness"]

results = []
for item in BENCHMARK:
    claim, label = item["claim"], item["label"]

    base_answer, base_evidence = baseline_answer(claim)
    base_faith = score_faithfulness_only(base_answer, base_evidence)
    base_correct = verdict_matches(base_answer, label)

    final_state = agent.invoke({"claim": claim, "iterations": 0})

    results.append({
        "claim": claim, "label": label,
        "baseline_answer": base_answer, "baseline_faithfulness": base_faith, "baseline_correct": base_correct,
        "agent_answer": final_state["answer"], "agent_faithfulness": final_state["faithfulness"],
        "agent_correct": verdict_matches(final_state["answer"], label),
        "agent_iterations": final_state["iterations"],
    })
    print(f"done: {claim[:50]}...  baseline={base_faith:.2f}  agent={final_state['faithfulness']:.2f}  iters={final_state['iterations']}")

print("\nevaluation complete")

In [ ]:
baseline_faith = [r["baseline_faithfulness"] for r in results]
agent_faith = [r["agent_faithfulness"] for r in results]
baseline_acc = np.mean([r["baseline_correct"] for r in results])
agent_acc = np.mean([r["agent_correct"] for r in results])
baseline_hallucination_rate = np.mean([f < FAITHFULNESS_THRESHOLD for f in baseline_faith])
agent_hallucination_rate = np.mean([f < FAITHFULNESS_THRESHOLD for f in agent_faith])
avg_iterations = np.mean([r["agent_iterations"] for r in results])

print(f"Baseline  -- accuracy: {baseline_acc:.0%}  avg faithfulness: {np.mean(baseline_faith):.2f}  hallucination rate: {baseline_hallucination_rate:.0%}")
print(f"Agent     -- accuracy: {agent_acc:.0%}  avg faithfulness: {np.mean(agent_faith):.2f}  hallucination rate: {agent_hallucination_rate:.0%}")
print(f"Avg iterations used by agent: {avg_iterations:.1f} / {MAX_ITERATIONS}")

## 7. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(["Baseline", "Agent"], [np.mean(baseline_faith), np.mean(agent_faith)], color=["#c0392b", "#27ae60"])
axes[0].set_title("Avg. Faithfulness Score")
axes[0].set_ylim(0, 1)

axes[1].bar(["Baseline", "Agent"], [baseline_acc, agent_acc], color=["#c0392b", "#27ae60"])
axes[1].set_title("Verdict Accuracy")
axes[1].set_ylim(0, 1)

axes[2].bar(["Baseline", "Agent"], [baseline_hallucination_rate, agent_hallucination_rate], color=["#c0392b", "#27ae60"])
axes[2].set_title("Hallucination Rate\n(faithfulness < threshold)")
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("faithfulness_comparison.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
x = np.arange(len(results))
plt.scatter(x, baseline_faith, label="Baseline", color="#c0392b")
plt.scatter(x, agent_faith, label="Agent (post-verification)", color="#27ae60")
plt.axhline(FAITHFULNESS_THRESHOLD, linestyle="--", color="gray", label="Threshold")
plt.xticks(x, [r["claim"][:20] + "..." for r in results], rotation=75, ha="right", fontsize=8)
plt.ylabel("Faithfulness score")
plt.title("Per-claim faithfulness: baseline vs. agent")
plt.legend()
plt.tight_layout()
plt.savefig("per_claim_faithfulness.png", dpi=150)
plt.show()

## 8. Detailed walkthrough of one example

Worth including in a README/demo: pick the claim where the agent's revision loop actually did
something, and show exactly what got flagged and fixed.

In [ ]:
revised_examples = [r for r in results if r["agent_iterations"] > 1]
example = revised_examples[0] if revised_examples else results[0]

print("CLAIM:", example["claim"])
print("\n--- BASELINE (single-shot, no verification) ---")
print(example["baseline_answer"])
print(f"faithfulness: {example['baseline_faithfulness']:.2f}")

print("\n--- AGENT (with verify-and-revise loop) ---")
print(example["agent_answer"])
print(f"faithfulness: {example['agent_faithfulness']:.2f}   iterations used: {example['agent_iterations']}")